In [1]:
import os
import tkinter as tk
from tkinter import filedialog, messagebox, ttk

import arcpy
from arcpy.sa import *

# ==========================================================
# ENVIRONMENT
# ==========================================================

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True


# ==========================================================
# FILE PICKERS
# ==========================================================

def select_file(title, initial_dir=None):
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    result = filedialog.askopenfilename(
        title=title,
        initialdir=initial_dir or ""
    )

    root.destroy()
    return result


def select_folder(title, initial_dir=None):
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    result = filedialog.askdirectory(
        title=title,
        initialdir=initial_dir or ""
    )

    root.destroy()
    return result


# ==========================================================
# LIST ALL FEATURE CLASSES
# ==========================================================

def list_all_feature_classes(gdb):
    arcpy.env.workspace = gdb

    fc_list = []

    root_fcs = arcpy.ListFeatureClasses()

    if root_fcs:
        for fc in root_fcs:
            fc_list.append(os.path.join(gdb, fc))

    datasets = arcpy.ListDatasets(feature_type="feature")

    if datasets:
        for ds in datasets:
            fcs = arcpy.ListFeatureClasses(feature_dataset=ds)

            if fcs:
                for fc in fcs:
                    fc_list.append(os.path.join(gdb, ds, fc))

    return fc_list


# ==========================================================
# FIND CONTOURS
# ==========================================================

def select_contours():
    print("\nSelect Geodatabase Containing Contours")

    gdb = select_folder("Select Geodatabase")

    if not gdb:
        raise RuntimeError("No geodatabase was selected.")

    fc_list = list_all_feature_classes(gdb)

    keywords = [
        "contour",
        "contours",
        "topo",
        "topography",
        "elevation"
    ]

    matches = []

    for fc in fc_list:
        name = os.path.basename(fc).lower()

        if any(k in name for k in keywords):
            matches.append(fc)

    if len(matches) == 1:
        print("\nUsing contour feature class:")
        print(matches[0])
        return matches[0]

    if len(matches) > 1:
        print("\nMultiple contour layers found:")
        for i, fc in enumerate(matches, 1):
            print(f"{i}: {fc}")

        choice = int(input("\nSelect number: "))
        return matches[choice - 1]

    raise RuntimeError("No contour feature class was found.")


class LeastCostWalkingGUI:
    def __init__(self):
        self.root = tk.Tk()
        self.root.title("Least Cost Walking Analysis")
        self.root.geometry("980x690")
        self.root.minsize(900, 620)
        self.root.configure(bg="#0f172a")

        self.style = ttk.Style(self.root)
        self.style.theme_use("clam")
        self.style.configure("TFrame", background="#0f172a")
        self.style.configure("TLabel", background="#0f172a", foreground="#e2e8f0", font=("Segoe UI", 10))
        self.style.configure("Header.TLabel", background="#0f172a", foreground="#7dd3fc", font=("Segoe UI", 22, "bold"))
        self.style.configure("Section.TLabel", background="#111827", foreground="#f8fafc", font=("Segoe UI", 11, "bold"))
        self.style.configure("TEntry", fieldbackground="#111827", foreground="#f8fafc", insertbackground="#f8fafc")
        self.style.configure("TButton", background="#1d4ed8", foreground="white", font=("Segoe UI", 10, "bold"))
        self.style.map("TButton", background=[("active", "#2563eb")])
        self.style.configure("Accent.TButton", background="#0ea5e9", foreground="white", font=("Segoe UI", 10, "bold"))
        self.style.map("Accent.TButton", background=[("active", "#0284c7")])

        self.values = {}
        self.fields = {
            "contours": tk.StringVar(),
            "elevation_field": tk.StringVar(),
            "start_points": tk.StringVar(),
            "end_points": tk.StringVar(),
            "id_field": tk.StringVar(),
            "output_folder": tk.StringVar(),
            "output_gdb_name": tk.StringVar(value="Walking_Analysis_Output.gdb"),
            "cell_size": tk.StringVar(value="10")
        }

        self.build_ui()

    def build_ui(self):
        outer = ttk.Frame(self.root, padding=(22, 18, 22, 18))
        outer.pack(fill="both", expand=True)

        heading = ttk.Label(outer, text="Least Cost Walking Analysis", style="Header.TLabel")
        heading.pack(anchor="w", pady=(0, 12))

        sub = ttk.Label(outer, text="ArcGIS Pro workflow setup", foreground="#cbd5e1")
        sub.pack(anchor="w", pady=(0, 16))

        card = ttk.Frame(outer, padding=(18, 16, 18, 16))
        card.pack(fill="both", expand=True)
        card.configure(style="TFrame")

        self._add_field(card, "Contours Feature Class", "contours", "Browse contour dataset", browse_callback=self.browse_contours)
        self._add_field(card, "Elevation Field Name", "elevation_field", "Example: GRID_CODE")
        self._add_field(card, "Start Points", "start_points", "Browse input start layer", browse_callback=self.browse_start_points)
        self._add_field(card, "End Points", "end_points", "Browse input end layer", browse_callback=self.browse_end_points)
        self._add_field(card, "Shared Route Field", "id_field", "Example: ROUTE_ID")
        self._add_field(card, "Output Folder", "output_folder", "Browse output workspace", browse_callback=self.browse_output_folder)
        self._add_field(card, "Output GDB Name", "output_gdb_name", "Walking_Analysis_Output.gdb")
        self._add_field(card, "DEM Cell Size", "cell_size", "10")

        button_row = ttk.Frame(card)
        button_row.pack(fill="x", pady=(18, 6))

        ttk.Button(button_row, text="Run Analysis", style="Accent.TButton", command=self.run_analysis).pack(side="right", padx=(8, 0))
        ttk.Button(button_row, text="Cancel", command=self.cancel).pack(side="right")

    def _add_field(self, parent, label_text, key, placeholder, browse_callback=None):
        row = ttk.Frame(parent, padding=(0, 6, 0, 0))
        row.pack(fill="x")

        label = ttk.Label(row, text=label_text, width=20, anchor="w")
        label.pack(side="left")

        entry = ttk.Entry(row, textvariable=self.fields[key], width=72)
        entry.insert(0, "")
        entry.pack(side="left", fill="x", expand=True, padx=(10, 8))

        if browse_callback:
            ttk.Button(row, text="Browse", command=lambda: browse_callback(key)).pack(side="left")

    def browse_contours(self, key):
        chosen = select_folder("Select Geodatabase Containing Contours")
        if chosen:
            self.fields[key].set(chosen)
            self._apply_contour_choice(chosen)

    def browse_start_points(self, key):
        chosen = select_file("Start Points")
        if chosen:
            self.fields[key].set(chosen)

    def browse_end_points(self, key):
        chosen = select_file("End Points")
        if chosen:
            self.fields[key].set(chosen)

    def browse_output_folder(self, key):
        chosen = select_folder("Output Folder")
        if chosen:
            self.fields[key].set(chosen)

    def _apply_contour_choice(self, gdb):
        if not gdb:
            return

        fc_list = list_all_feature_classes(gdb)
        keywords = ["contour", "contours", "topo", "topography", "elevation"]
        matches = []

        for fc in fc_list:
            name = os.path.basename(fc).lower()
            if any(k in name for k in keywords):
                matches.append(fc)

        if len(matches) == 0:
            messagebox.showwarning("Contour Selection", "No contour feature class was found in the selected geodatabase.")
            return

        if len(matches) == 1:
            self.fields["contours"].set(matches[0])
            return

        chooser = tk.Toplevel(self.root)
        chooser.title("Choose Contour Feature Class")
        chooser.geometry("600x360")
        chooser.configure(bg="#0f172a")
        chooser.transient(self.root)
        chooser.grab_set()

        ttk.Label(chooser, text="Multiple contour layers were found. Choose one:", foreground="#f8fafc").pack(anchor="w", padx=16, pady=(16, 10))

        listbox = tk.Listbox(chooser, bg="#111827", fg="#f8fafc", selectbackground="#0ea5e9", height=12, width=90, borderwidth=0)
        listbox.pack(fill="both", expand=True, padx=16, pady=(0, 12))

        for fc in matches:
            listbox.insert(tk.END, fc)

        def _set_choice():
            selection = listbox.curselection()
            if selection:
                self.fields["contours"].set(matches[selection[0]])
            chooser.destroy()

        ttk.Button(chooser, text="Select", command=_set_choice).pack(anchor="e", padx=16, pady=(0, 16))
        chooser.wait_window()

    def run_analysis(self):
        required_keys = [
            "contours",
            "elevation_field",
            "start_points",
            "end_points",
            "id_field",
            "output_folder",
            "output_gdb_name",
            "cell_size"
        ]

        for key in required_keys:
            if not self.fields[key].get().strip():
                messagebox.showerror("Missing value", f"Please provide a value for: {key.replace('_', ' ').title()}")
                return

        try:
            float(self.fields["cell_size"].get())
        except ValueError:
            messagebox.showerror("Invalid Cell Size", "DEM cell size must be a numeric value.")
            return

        self.values = {key: self.fields[key].get().strip() for key in required_keys}
        self.root.destroy()

    def cancel(self):
        self.values = None
        self.root.destroy()

    def show(self):
        self.root.mainloop()
        return self.values


print("\n====================================")
print("Least Cost Walking Path Tool")
print("====================================")

app = LeastCostWalkingGUI()
inputs = app.show()

if inputs is None:
    raise RuntimeError("Analysis cancelled by user.")

contours = inputs["contours"]
elevation_field = inputs["elevation_field"]
start_points = inputs["start_points"]
end_points = inputs["end_points"]
id_field = inputs["id_field"]
output_folder = inputs["output_folder"]
output_gdb_name = inputs["output_gdb_name"]

if not output_gdb_name.lower().endswith(".gdb"):
    output_gdb_name += ".gdb"

output_gdb = os.path.join(output_folder, output_gdb_name)

if not arcpy.Exists(output_gdb):
    arcpy.management.CreateFileGDB(output_folder, output_gdb_name)

cell_size = float(inputs["cell_size"])

# ==========================================================
# DEM
# ==========================================================

dem = os.path.join(output_gdb, "DEM")

if arcpy.Exists(dem):
    print("\nDEM already exists.")
else:
    print("\nCreating DEM...")

    topo_input = [[contours, elevation_field, "CONTOUR"]]
    dem_raster = arcpy.sa.TopoToRaster(topo_input, cell_size)
    dem_raster.save(dem)

    print("DEM creation is complete.")

# ==========================================================
# SLOPE
# ==========================================================

slope_output = os.path.join(output_gdb, "Slope_Degrees")

if arcpy.Exists(slope_output):
    print("\nSlope output raster already exists.")
else:
    print("\nCreating Slope...")

    slope_raster = Slope(dem, output_measurement="DEGREE")
    slope_raster.save(slope_output)

    print("Slope raster is complete.")

# ==========================================================
# WALKING COST
# ==========================================================

cost_raster = os.path.join(output_gdb, "Walking_Cost")

if arcpy.Exists(cost_raster):
    print("\nWalking Cost Pathways already exists.")
else:
    print("\nCreating Walking Cost Pathways...")

    walking_cost = Reclassify(
        slope_output,
        "VALUE",
        RemapRange([
            [0, 5, 1],
            [5, 10, 2],
            [10, 15, 4],
            [15, 20, 8],
            [20, 30, 15],
            [30, 45, 50],
            [45, 90, 100]
        ])
    )

    walking_cost.save(cost_raster)

    print("Least Cost Walking Pathways is complete.")

# ==========================================================
# READ ROUTES
# ==========================================================

route_ids = []

with arcpy.da.SearchCursor(start_points, [id_field]) as cursor:
    for row in cursor:
        if row[0] is not None:
            route_ids.append(str(row[0]))

route_ids = sorted(list(set(route_ids)))

print(f"\nFound {len(route_ids)} routes.")
print("\nSample Labels:")
for x in route_ids[:10]:
    print(repr(x))

# ==========================================================
# CREATE LAYERS
# ==========================================================

arcpy.management.MakeFeatureLayer(start_points, "start_lyr")
arcpy.management.MakeFeatureLayer(end_points, "end_lyr")

# ==========================================================
# PROCESS ROUTES
# ==========================================================

route_outputs = []

for route_id in route_ids:
    print(f"\nProcessing: {route_id}")

    safe_name = (
        str(route_id)
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace("-", "_")
        .replace(":", "_")
        .replace(".", "_")
    )

    route_fc = os.path.join(output_gdb, f"Route_{safe_name}")

    if arcpy.Exists(route_fc):
        print("Route already exists. Skipping.")
        route_outputs.append(route_fc)
        continue

    field_delim = arcpy.AddFieldDelimiters(start_points, id_field)
    route_value = str(route_id).replace("'", "''")
    where_clause = f"{field_delim} = '{route_value}'"

    print(f"SQL: {where_clause}")

    arcpy.management.SelectLayerByAttribute("start_lyr", "NEW_SELECTION", where_clause)
    arcpy.management.SelectLayerByAttribute("end_lyr", "NEW_SELECTION", where_clause)

    start_count = int(arcpy.management.GetCount("start_lyr")[0])
    end_count = int(arcpy.management.GetCount("end_lyr")[0])

    print(f"Start={start_count} End={end_count}")

    if start_count == 0:
        print("No matching starting point.")
        continue

    if end_count == 0:
        print("No matching ending point.")
        continue

    distance_raster = os.path.join(output_gdb, f"Distance_{safe_name}")
    backdir_raster = os.path.join(output_gdb, f"BackDir_{safe_name}")

    if arcpy.Exists(distance_raster) and arcpy.Exists(backdir_raster):
        print("Using existing accumulation rasters.")
    else:
        print("Running Distance Accumulation Analysis...")
        dist_result = DistanceAccumulation(
            in_source_data="start_lyr",
            in_cost_raster=cost_raster,
            out_back_direction_raster=backdir_raster
        )

        dist_result.save(distance_raster)

    print("Creating Least Cost Path...")

    try:
        OptimalPathAsLine(
            in_destination_data="end_lyr",
            in_distance_accumulation_raster=distance_raster,
            in_back_direction_raster=backdir_raster,
            out_polyline_features=route_fc,
            destination_field=arcpy.Describe(end_points).OIDFieldName,
            path_type="BEST_SINGLE"
        )

        route_outputs.append(route_fc)
        print(f"Route complete: {route_id}")
    except Exception as e:
        print(f"Route failed: {route_id}")
        print(str(e))
        continue

# ==========================================================
# MERGE ROUTES
# ==========================================================

merged_routes = os.path.join(output_gdb, "LeastCostRoutes")

if arcpy.Exists(merged_routes):
    print("\nDeleting old LeastCostRoutes...")
    arcpy.management.Delete(merged_routes)

print("\nMerging Routes...")
arcpy.management.Merge(route_outputs, merged_routes)

# ==========================================================
# FINISHED
# ==========================================================

print(f"\nRoutes Processed: {len(route_outputs)}")
print("\n====================================")
print("PROCESS COMPLETE")
print("====================================")
print(f"\nOutput GDB: {output_gdb}")
print(f"Final Routes: {merged_routes}")



Least Cost Walking Path Tool

DEM already exists.

Slope output raster already exists.

Walking Cost Pathways already exists.

Found 10 routes.

Sample Labels:
'Point A'
'Point B'
'Point C'
'Point D'
'Point E'
'Point F'
'Point G'
'Point H'
'Point I'
'Point J'

Processing: Point A
Route already exists. Skipping.

Processing: Point B
Route already exists. Skipping.

Processing: Point C
Route already exists. Skipping.

Processing: Point D
Route already exists. Skipping.

Processing: Point E
Route already exists. Skipping.

Processing: Point F
Route already exists. Skipping.

Processing: Point G
Route already exists. Skipping.

Processing: Point H
Route already exists. Skipping.

Processing: Point I
Route already exists. Skipping.

Processing: Point J
Route already exists. Skipping.

Deleting old LeastCostRoutes...

Merging Routes...

Routes Processed: 10

PROCESS COMPLETE

Output GDB: C:/Users/ClarenceBodmer4L/OneDrive - 4Liberty/ArcGIS Pro Projects/Projects/Mountaintop_Microwave_Locatio